# Netflix Content Trends Analysis — Strategic Recommendations

> Problem: Analyze how Netflix’s content distribution (Movies vs TV Shows), genres, and country contributions have evolved over the years to inform strategy.

This notebook explores:
- Distribution and growth of Movies vs. TV Shows over time
- Most frequent genres and their popularity trends
- Country-wise contributions to Netflix’s catalog
- Strategic recommendations for acquisition and production

This notebook is configured to use your local dataset strictly (no online fallbacks). Place `Netflix_Dataset.csv` in the project folder and run the cells top-to-bottom.

## Objectives
- Quantify growth of Movies vs TV Shows by year
- Identify top genres and observe how popularity shifts over time
- Compare country contributions to assess global diversity
- Translate findings into practical strategy recommendations

## Significance
Insights help Netflix:
- Balance global vs. local productions
- Recognize market trends and regional interests
- Make data-driven investment decisions in genres and countries

## Tech Stack
- Python, Jupyter/Colab
- pandas, numpy, matplotlib, seaborn, plotly

In [ ]:
# Install dependencies if missing (works in Jupyter/Colab)
import sys, subprocess
def ensure(pkg):
    try:
        __import__(pkg)
    except Exception:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])
        except Exception as e:
            print(f'Warning: Could not install {pkg}: {e}')

for p in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'plotly', 'pycountry']:
    ensure(p)
# country_converter is optional for the map; skip if not available
try:
    __import__('country_converter')
except Exception:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'country_converter'])
    except Exception as e:
        print(f'Note: country_converter install failed (map will be optional): {e}')

In [ ]:
# Imports and settings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import datetime

# Use project module for robust, local-only loading
from src.analysis import load_netflix_dataset, clean_and_engineer

pd.set_option('display.max_colwidth', 120)
sns.set(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)

# Default dataset path (your local file in project folder)
DATA_PATH = 'Netflix_Dataset.csv'

In [ ]:
# Load dataset strictly from local project folder (no mirrors/sample)
import os

df_raw = load_netflix_dataset(DATA_PATH, strict=True)
df_raw.head()

In [ ]:
# Basic overview
(df_raw.shape, df_raw.dtypes)

In [ ]:
# Cleaning & feature engineering using our robust function
from src.analysis import clean_and_engineer

df = clean_and_engineer(df_raw)

# Keep only rows with year info for trend analyses
df_year = df.dropna(subset=['year_added']).copy()
if not df_year.empty:
    df_year['year_added'] = df_year['year_added'].astype(int)

df.head()

In [ ]:
# Missing values summary
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

## Movies vs. TV Shows — Overall and Over Time
We first compare counts and then visualize how each type grew by year.

In [ ]:
# Overall distribution
ax = sns.countplot(data=df, x='type', order=['Movie', 'TV Show'], palette='Set2')
ax.bar_label(ax.containers[0])
plt.title('Overall Distribution: Movies vs TV Shows')
plt.xlabel('Type'); plt.ylabel('Count')
plt.show()

In [ ]:
# Annual additions by type
annual = (df_year.groupby(['year_added', 'type'])
                .size()
                .reset_index(name='count'))

fig, ax = plt.subplots(figsize=(12,5))
sns.lineplot(data=annual, x='year_added', y='count', hue='type', marker='o', ax=ax)
plt.title('Additions per Year by Type')
plt.xlabel('Year added'); plt.ylabel('Titles added')
plt.tight_layout(); plt.show()

annual.tail()

## Genres — Frequency and Trends
Genres come from the `listed_in` field; we split and normalize labels.

In [ ]:
# Explode genres using shared normalizer for consistent categories
from src.analysis import explode_and_normalize_genres

genres = explode_and_normalize_genres(df)

# Top genres overall (Top 15)
top_genres = (genres['genre'].value_counts().head(15))
ax = top_genres.sort_values().plot(kind='barh', color='#1f77b4', figsize=(10,6))
plt.title('Top 15 Genres (Overall) — Normalized')
plt.xlabel('Count'); plt.ylabel('Genre')
plt.tight_layout(); plt.show()
top_genres

In [ ]:
# Genre trends over years (for top 10)
focus_genres = top_genres.head(10).index.tolist()
genre_year = (genres.dropna(subset=['year_added'])
                .query('genre in @focus_genres')
                .groupby(['year_added', 'genre'])
                .size()
                .reset_index(name='count'))

fig, ax = plt.subplots(figsize=(12,6))
sns.lineplot(data=genre_year, x='year_added', y='count', hue='genre', marker='o', ax=ax)
plt.title('Genre Popularity Over Time (Top 10 Genres)')
plt.xlabel('Year added'); plt.ylabel('Titles added')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()
genre_year.tail()

## Country Contributions
We compute contributions by exploding the `country` column and counting titles per country.

In [ ]:
# Explode countries
countries = (df[['show_id', 'country', 'year_added']]
             .assign(country=lambda d: d['country'].str.split(','))
             .explode('country'))
countries['country'] = countries['country'].astype(str).str.strip()
countries = countries.replace({'country': {'': 'Unknown', 'nan': 'Unknown'}})

top_countries = countries['country'].value_counts().head(15)
ax = top_countries.sort_values().plot(kind='barh', color='#ff7f0e', figsize=(10,6))
plt.title('Top 15 Contributing Countries (Overall)')
plt.xlabel('Count'); plt.ylabel('Country')
plt.tight_layout(); plt.show()
top_countries

In [ ]:
# Optional: Choropleth map of contributions (requires country_converter)
try:
    import country_converter as coco
    cc = coco.CountryConverter()
    country_counts = countries.groupby('country').size().reset_index(name='count')
    country_counts['iso3'] = cc.convert(country_counts['country'], to='ISO3', not_found=None)
    cc_map = country_counts.dropna(subset=['iso3'])
    fig = px.choropleth(cc_map, locations='iso3', color='count', color_continuous_scale='Reds',
                        title='Country Contributions (Choropleth)')
    fig.show()
except Exception as e:
    print('Choropleth map skipped:', e)

## 5. Content Rating Distribution

In [ ]:
import plotly.express as px

# Content Rating Distribution
rating_counts = df['rating'].value_counts().head(15).reset_index()
rating_counts.columns = ['rating', 'count']

fig = px.pie(rating_counts, values='count', names='rating', 
             title='Content Rating Distribution',
             color_discrete_sequence=px.colors.qualitative.Pastel)
fig.show()

# Bar chart alternative
plt.figure(figsize=(12, 6))
sns.barplot(data=rating_counts.head(10), x='count', y='rating', palette='Set2')
plt.title('Top 10 Content Ratings on Netflix', fontsize=14, fontweight='bold')
plt.xlabel('Count')
plt.ylabel('Rating')
plt.tight_layout()
plt.show()

## 6. Top Directors Analysis

In [ ]:
# Top Directors by Content Count
directors = (df[['show_id', 'director']]
            .assign(director=lambda d: d['director'].str.split(','))
            .explode('director'))
directors['director'] = directors['director'].astype(str).str.strip()
directors = directors[directors['director'].notna() & 
                     (directors['director'] != 'Unknown') & 
                     (directors['director'] != 'nan')]

top_directors = directors['director'].value_counts().head(15).reset_index()
top_directors.columns = ['director', 'count']

# Plotly bar chart
fig = px.bar(top_directors.sort_values('count'), y='director', x='count', 
             orientation='h', color='count', 
             color_continuous_scale='Greens',
             title='Top 15 Most Prolific Directors on Netflix')
fig.show()

# Matplotlib alternative
plt.figure(figsize=(12, 8))
sns.barplot(data=top_directors.sort_values('count', ascending=True), 
            y='director', x='count', palette='Greens_r')
plt.title('Top 15 Directors by Content Count', fontsize=14, fontweight='bold')
plt.xlabel('Number of Titles')
plt.ylabel('Director')
plt.tight_layout()
plt.show()

## 7. Movie Duration Analysis

In [ ]:
# Movie Duration Distribution
movies_only = df[df['type'] == 'Movie'].copy()
movies_only['duration_min'] = movies_only['duration'].astype(str).str.extract(r'(\d+)').astype(float)
movies_only = movies_only.dropna(subset=['duration_min'])

if not movies_only.empty:
    # Plotly histogram
    fig = px.histogram(movies_only, x='duration_min', nbins=30,
                      title='Movie Duration Distribution',
                      labels={'duration_min': 'Duration (minutes)', 'count': 'Number of Movies'},
                      color_discrete_sequence=['#1f77b4'])
    fig.show()
    
    # Matplotlib histogram with stats
    plt.figure(figsize=(12, 6))
    plt.hist(movies_only['duration_min'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    plt.axvline(movies_only['duration_min'].mean(), color='red', linestyle='--', 
                linewidth=2, label=f"Mean: {movies_only['duration_min'].mean():.1f} min")
    plt.axvline(movies_only['duration_min'].median(), color='green', linestyle='--', 
                linewidth=2, label=f"Median: {movies_only['duration_min'].median():.1f} min")
    plt.title('Movie Duration Distribution with Statistics', fontsize=14, fontweight='bold')
    plt.xlabel('Duration (minutes)')
    plt.ylabel('Number of Movies')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"Movie Duration Statistics:")
    print(f"  Mean: {movies_only['duration_min'].mean():.1f} minutes")
    print(f"  Median: {movies_only['duration_min'].median():.1f} minutes")
    print(f"  Min: {movies_only['duration_min'].min():.0f} minutes")
    print(f"  Max: {movies_only['duration_min'].max():.0f} minutes")
else:
    print("No movie duration data available.")

## 8. TV Show Seasons Distribution

In [ ]:
# TV Show Seasons Distribution
tv_only = df[df['type'] == 'TV Show'].copy()
tv_only['seasons'] = tv_only['duration'].astype(str).str.extract(r'(\d+)').astype(float)
tv_only = tv_only.dropna(subset=['seasons'])

if not tv_only.empty:
    season_counts = tv_only['seasons'].value_counts().sort_index().head(10).reset_index()
    season_counts.columns = ['seasons', 'count']
    
    # Plotly bar chart
    fig = px.bar(season_counts, x='seasons', y='count',
                title='TV Shows by Number of Seasons',
                labels={'seasons': 'Number of Seasons', 'count': 'Number of Shows'},
                color='count', color_continuous_scale='Reds')
    fig.show()
    
    # Matplotlib bar chart
    plt.figure(figsize=(12, 6))
    sns.barplot(data=season_counts, x='seasons', y='count', palette='Reds_r')
    plt.title('Distribution of TV Shows by Season Count', fontsize=14, fontweight='bold')
    plt.xlabel('Number of Seasons')
    plt.ylabel('Number of Shows')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    print(f"TV Show Season Statistics:")
    print(f"  Most common: {tv_only['seasons'].mode().values[0]:.0f} season(s)")
    print(f"  Mean: {tv_only['seasons'].mean():.1f} seasons")
    print(f"  Max: {tv_only['seasons'].max():.0f} seasons")
    
    # Count of single-season vs multi-season
    single_season = (tv_only['seasons'] == 1).sum()
    multi_season = (tv_only['seasons'] > 1).sum()
    print(f"  Single-season shows: {single_season} ({single_season/len(tv_only)*100:.1f}%)")
    print(f"  Multi-season shows: {multi_season} ({multi_season/len(tv_only)*100:.1f}%)")
else:
    print("No TV show season data available.")

## 9. Genre Trends Over Time

In [ ]:
# Genre Trends Over Time - Top 5 Genres
gdf_time = explode_and_normalize_genres(df)
gdf_time = gdf_time.dropna(subset=['year_added'])

if not gdf_time.empty:
    # Get top 5 overall genres
    top5_genres = gdf_time['genre'].value_counts().head(5).index.tolist()
    print(f"Top 5 Genres: {', '.join(top5_genres)}")
    
    gdf_top5 = gdf_time[gdf_time['genre'].isin(top5_genres)]
    genre_year = gdf_top5.groupby(['year_added', 'genre']).size().reset_index(name='count')
    
    # Plotly line chart
    fig = px.line(genre_year, x='year_added', y='count', color='genre',
                 markers=True, title='Top 5 Genre Evolution Over Years')
    fig.update_layout(xaxis_title='Year Added', yaxis_title='Number of Titles')
    fig.show()
    
    # Matplotlib line chart
    plt.figure(figsize=(14, 7))
    for genre in top5_genres:
        genre_data = genre_year[genre_year['genre'] == genre]
        plt.plot(genre_data['year_added'], genre_data['count'], 
                marker='o', linewidth=2, label=genre)
    
    plt.title('Genre Popularity Trends Over Time (Top 5)', fontsize=14, fontweight='bold')
    plt.xlabel('Year Added')
    plt.ylabel('Number of Titles')
    plt.legend(title='Genre', loc='best')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No year data available for genre trends.")

## 10. Monthly Addition Patterns (Seasonality)

In [ ]:
# Monthly Addition Patterns
df_with_dates = df.dropna(subset=['date_added']).copy()

if not df_with_dates.empty:
    try:
        df_with_dates['month'] = pd.to_datetime(df_with_dates['date_added'], errors='coerce').dt.month_name()
        month_order = ['January', 'February', 'March', 'April', 'May', 'June',
                      'July', 'August', 'September', 'October', 'November', 'December']
        month_counts = df_with_dates['month'].value_counts().reindex(month_order).reset_index()
        month_counts.columns = ['month', 'count']
        
        # Plotly bar chart
        fig = px.bar(month_counts, x='month', y='count',
                    title='Content Additions by Month (All Years)',
                    color='count', color_continuous_scale='Viridis')
        fig.update_xaxes(tickangle=45)
        fig.show()
        
        # Matplotlib bar chart
        plt.figure(figsize=(14, 6))
        sns.barplot(data=month_counts, x='month', y='count', palette='viridis')
        plt.title('Seasonal Content Addition Patterns', fontsize=14, fontweight='bold')
        plt.xlabel('Month')
        plt.ylabel('Number of Titles Added')
        plt.xticks(rotation=45, ha='right')
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        # Find peak months
        peak_month = month_counts.loc[month_counts['count'].idxmax(), 'month']
        peak_count = month_counts['count'].max()
        print(f"Peak addition month: {peak_month} with {peak_count} titles")
        print(f"Average monthly additions: {month_counts['count'].mean():.0f} titles")
    except Exception as e:
        print(f"Unable to parse month data: {e}")
else:
    print("No date_added data available.")

## Automated Insights Snapshot
Quick, data-driven indicators based on current results.

In [ ]:
# Generate a few simple insights
insights = []

# 1) Latest year momentum
if 'annual' in globals() and not annual.empty:
    latest_year = int(annual['year_added'].max())
    latest = annual[annual['year_added'] == latest_year].sort_values('count', ascending=False)
    lines = [f"  - {t}: {c}" for t, c in zip(latest['type'], latest['count'])]
    msg = 'Latest year with data: ' + str(latest_year) + '\n' + "\n".join(lines)
    insights.append('Annual additions — ' + msg)

# 2) Top overall genres (top 5)
if 'top_genres' in globals() and top_genres is not None and len(top_genres) > 0:
    gmsg = ', '.join([f"{g} ({n})" for g, n in top_genres.head(5).items()])
    insights.append('Top genres overall: ' + gmsg)

# 3) Top contributing countries (top 5)
if 'top_countries' in globals() and top_countries is not None and len(top_countries) > 0:
    cmsg = ', '.join([f"{c} ({n})" for c, n in top_countries.head(5).items()])
    insights.append('Top contributing countries: ' + cmsg)

print("\nINSIGHTS SUMMARY\n-----------------\n" + "\n".join(insights) if insights else "No insights available yet — run previous cells.")